# Imports

In [ ]:
# Adds the parent directory to sys.path
import sys

sys.path.append('../')

import pickle
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.colors as pc
import plotly.graph_objects as go
from IPython.display import display
from problems.benchmark_problems import get_pareto
from problems.microgrid_function import simulation
from pygmo import fast_non_dominated_sorting  # type: ignore


# Microgrid planning results analysis

This section reconstructs the complete battery-specific non-dominated fronts from the 30 independent runs, forms their global non-dominated union, applies TOPSIS with adjustable weights, and prepares the tables and figures used in the article. The original exploratory plotting code is preserved in the next major section.

## 1. Analysis configuration

The reported criteria are LCOE (cost), RF (benefit), and RSC (benefit). Equal TOPSIS weights are used initially, but the vector can be changed without modifying the remaining analysis.

In [ ]:
RESULTS_DIR = Path('../results')

BATTERY_LABELS = {
    'Lead_Acid': 'Lead-acid',
    'Li-ion': 'Li-ion',
    'ZEBRA': 'ZEBRA',
    'NaS': 'NaS',
    'NiCd': 'NiCd',
    'NiMH': 'NiMH',
    'RFV': 'VRF',
    'ZnBr': 'ZnBr',
}

POSITION_COLUMNS = ['PV rated power', 'Wind rated power', 'Battery capacity']
CRITERIA_COLUMNS = ['LCOE', 'RF', 'RSC']
CRITERIA_DIRECTIONS = np.array(['min', 'max', 'max'])

n_criteria = len(CRITERIA_COLUMNS)
TOPSIS_WEIGHTS = np.full(n_criteria, 1 / n_criteria)

BATTERY_COLORS = {
    code: mcolors.to_hex(plt.get_cmap('tab10')(index))
    for index, code in enumerate(BATTERY_LABELS)
}


## 2. Front-construction and TOPSIS functions

The result files store RF and RSC with negative signs because A-MESH uses a minimization convention. The loading function restores their reported positive values before constructing the data frames. For non-dominated sorting, RF and RSC are temporarily converted back to minimization form.

In [ ]:
def validate_directions(directions, n_criteria):
    """Validate and standardize criterion directions."""
    directions = np.char.lower(np.asarray(directions, dtype=str))
    if directions.shape != (n_criteria,):
        raise ValueError('directions must contain one entry per criterion.')
    if not np.all(np.isin(directions, ['min', 'max'])):
        raise ValueError("directions must contain only 'min' or 'max'.")
    return directions


def to_minimization(criteria, directions):
    """Convert reported criteria to an all-minimization representation."""
    criteria = np.asarray(criteria, dtype=float)
    if criteria.ndim != 2:
        raise ValueError('criteria must be a two-dimensional array.')
    directions = validate_directions(directions, criteria.shape[1])
    signs = np.where(directions == 'min', 1.0, -1.0)
    return criteria * signs


def from_minimization(stored_objectives, directions):
    """Restore reported values from objectives stored for minimization."""
    stored_objectives = np.asarray(stored_objectives, dtype=float)
    if stored_objectives.ndim != 2:
        raise ValueError('stored_objectives must be a two-dimensional array.')
    directions = validate_directions(directions, stored_objectives.shape[1])
    signs = np.where(directions == 'min', 1.0, -1.0)
    return stored_objectives * signs


def nondominated_indices(criteria, directions):
    """Return indices of the first non-dominated front."""
    criteria = np.asarray(criteria, dtype=float)
    if criteria.ndim != 2:
        raise ValueError('criteria must be a two-dimensional array.')
    validate_directions(directions, criteria.shape[1])
    if len(criteria) == 0:
        return np.empty(0, dtype=int)
    if len(criteria) == 1:
        return np.array([0], dtype=int)

    minimization_values = to_minimization(criteria, directions)
    return np.asarray(
        fast_non_dominated_sorting(points=minimization_values)[0][0],
        dtype=int,
    )


def load_battery_front(
    battery_code,
    position_columns=POSITION_COLUMNS,
    criteria_columns=CRITERIA_COLUMNS,
    directions=CRITERIA_DIRECTIONS,
    results_dir=RESULTS_DIR,
):
    """Reconstruct one complete technology-specific front."""
    n_positions = len(position_columns)
    n_criteria = len(criteria_columns)
    directions = validate_directions(directions, n_criteria)
    file_path = results_dir / f'AMESH_{battery_code}_{n_criteria}_{n_positions}.pkl'
    if not file_path.exists():
        raise FileNotFoundError(
            f'Result file not found: {file_path}. '
            'Place the microgrid result files in scripts/results.'
        )

    with file_path.open('rb') as file:
        stored_results = pickle.load(file)

    run_keys = sorted(key for key in stored_results if isinstance(key, int))
    if not run_keys:
        raise ValueError(f'No independent runs were found in {file_path}.')

    positions = np.vstack([stored_results[key]['P'] for key in run_keys])
    stored_objectives = np.vstack([stored_results[key]['F'] for key in run_keys])

    # The simulation is deterministic for a fixed decision vector. Keeping the
    # first occurrence therefore removes repeated alternatives without changing
    # their objective values.
    unique_positions, unique_indices = np.unique(
        positions, axis=0, return_index=True
    )
    unique_objectives = stored_objectives[unique_indices]
    if unique_positions.shape[1] != n_positions:
        raise ValueError(f'{file_path} does not match position_columns.')
    if unique_objectives.shape[1] != n_criteria:
        raise ValueError(f'{file_path} does not match criteria_columns.')

    reported_criteria = from_minimization(unique_objectives, directions)
    front_indices = nondominated_indices(reported_criteria, directions)

    front = pd.DataFrame(
        np.column_stack(
            (unique_positions[front_indices], reported_criteria[front_indices])
        ),
        columns=list(position_columns) + list(criteria_columns),
    )
    front.insert(0, 'Battery code', battery_code)
    front.insert(1, 'Battery', BATTERY_LABELS[battery_code])

    statistics = {
        'Battery': BATTERY_LABELS[battery_code],
        'Runs': len(run_keys),
        'Pooled alternatives': len(positions),
        'Unique alternatives': len(unique_positions),
        'Specific front': len(front),
    }
    return front.reset_index(drop=True), statistics


def topsis_rank(front, criteria_columns, weights, directions):
    """Rank alternatives by TOPSIS for any number of criteria."""
    criteria_columns = list(criteria_columns)
    n_criteria = len(criteria_columns)
    weights = np.asarray(weights, dtype=float)
    directions = validate_directions(directions, n_criteria)
    if weights.shape != (n_criteria,):
        raise ValueError('weights must contain one value for each criterion.')
    if np.any(weights < 0) or not np.isclose(weights.sum(), 1.0):
        raise ValueError('weights must be nonnegative and sum to one.')
    if front.empty:
        raise ValueError('TOPSIS cannot rank an empty front.')

    decision_matrix = front[criteria_columns].to_numpy(dtype=float)
    column_norms = np.linalg.norm(decision_matrix, axis=0)
    safe_norms = np.where(np.isclose(column_norms, 0.0), 1.0, column_norms)
    weighted_matrix = decision_matrix / safe_norms * weights
    ideal = np.empty(n_criteria)
    negative_ideal = np.empty(n_criteria)
    for column, direction in enumerate(directions):
        if direction == 'min':
            ideal[column] = weighted_matrix[:, column].min()
            negative_ideal[column] = weighted_matrix[:, column].max()
        else:
            ideal[column] = weighted_matrix[:, column].max()
            negative_ideal[column] = weighted_matrix[:, column].min()

    distance_to_ideal = np.linalg.norm(weighted_matrix - ideal, axis=1)
    distance_to_negative_ideal = np.linalg.norm(
        weighted_matrix - negative_ideal, axis=1
    )
    distance_sum = distance_to_ideal + distance_to_negative_ideal
    closeness = np.divide(
        distance_to_negative_ideal,
        distance_sum,
        out=np.full_like(distance_sum, 0.5),
        where=~np.isclose(distance_sum, 0.0),
    )

    ranked = front.copy()
    ranked['Distance to ideal'] = distance_to_ideal
    ranked['Distance to negative ideal'] = distance_to_negative_ideal
    ranked['TOPSIS coefficient'] = closeness
    ranked['TOPSIS rank'] = (
        ranked['TOPSIS coefficient']
        .rank(method='min', ascending=False)
        .astype(int)
    )
    return ranked.sort_values(
        'TOPSIS coefficient', ascending=False, kind='mergesort'
    ).reset_index(drop=True)


## 3. Battery-specific and global non-dominated fronts

A point dominated within its own battery-specific set cannot become globally non-dominated. Therefore, the global front can be obtained efficiently from the union of the eight complete technology-specific fronts. No crowding-distance truncation is applied before the decision analysis.

In [ ]:
specific_fronts = {}
front_statistics = []

for battery_code in BATTERY_LABELS:
    front, statistics = load_battery_front(
        battery_code,
        position_columns=POSITION_COLUMNS,
        criteria_columns=CRITERIA_COLUMNS,
        directions=CRITERIA_DIRECTIONS,
    )
    specific_fronts[battery_code] = front
    front_statistics.append(statistics)

specific_union = pd.concat(specific_fronts.values(), ignore_index=True)
global_indices = nondominated_indices(
    specific_union[CRITERIA_COLUMNS].to_numpy(), CRITERIA_DIRECTIONS
)
global_front = specific_union.iloc[global_indices].reset_index(drop=True)

front_statistics = pd.DataFrame(front_statistics)
global_contributions = (
    global_front.groupby('Battery', sort=False)
    .size()
    .rename('Global front')
)
front_statistics = (
    front_statistics.set_index('Battery')
    .join(global_contributions, how='left')
    .fillna({'Global front': 0})
)
front_statistics['Global front'] = front_statistics['Global front'].astype(int)
display(front_statistics)
print(f'Global non-dominated alternatives: {len(global_front)}')


## 4. TOPSIS compromise solutions

TOPSIS is first applied separately to each battery-specific front and then to the global front. Equal weights are adopted at this stage. A coefficient obtained from one specific front must not be compared directly with a coefficient obtained from another because each ranking uses its own ideal and negative-ideal solutions. The global ranking is the appropriate basis for comparison among battery technologies.

In [ ]:
specific_rankings = {
    battery_code: topsis_rank(
        front, CRITERIA_COLUMNS, TOPSIS_WEIGHTS, CRITERIA_DIRECTIONS
    )
    for battery_code, front in specific_fronts.items()
}
global_ranking = topsis_rank(
    global_front, CRITERIA_COLUMNS, TOPSIS_WEIGHTS, CRITERIA_DIRECTIONS
)

selected_rows = []
for battery_code, ranking in specific_rankings.items():
    selected = ranking.iloc[0].copy()
    selected['Scope'] = BATTERY_LABELS[battery_code]
    selected_rows.append(selected)

global_selected = global_ranking.iloc[0].copy()
global_selected['Scope'] = 'Global'
selected_rows.append(global_selected)

compromise_table = pd.DataFrame(selected_rows)[
    [
        'Scope',
        'Battery',
        *POSITION_COLUMNS,
        *CRITERIA_COLUMNS,
        'TOPSIS coefficient',
    ]
]

display(
    compromise_table.style.format(
        {
            'PV rated power': '{:.2f}',
            'Wind rated power': '{:.2f}',
            'Battery capacity': '{:.2f}',
            'LCOE': '{:.4f}',
            'RF': '{:.4f}',
            'RSC': '{:.4f}',
            'TOPSIS coefficient': '{:.4f}',
        }
    )
)



def dataframe_to_latex_table(
    frame,
    caption,
    label,
    column_labels=None,
    formats=None,
    column_format=None,
    environment='table*',
):
    """Return a compact booktabs table ready to paste into the article."""
    column_labels = column_labels or {}
    formats = formats or {}
    displayed = frame.rename(columns=column_labels)
    formatters = {
        column_labels.get(column, column): (
            lambda value, specification=specification: format(value, specification)
        )
        for column, specification in formats.items()
    }
    if column_format is None:
        column_format = 'l' + 'r' * (len(displayed.columns) - 1)

    tabular = displayed.to_latex(
        index=False,
        escape=False,
        formatters=formatters,
        column_format=column_format,
    ).strip()
    return '\n'.join(
        [
            rf'\begin{{{environment}}}[!t]',
            r'    \centering',
            rf'    \caption{{{caption}}}',
            rf'    \label{{{label}}}',
            r'    \scriptsize',
            r'    \setlength{\tabcolsep}{3pt}',
            r'    \renewcommand{\arraystretch}{1.05}',
            tabular,
            rf'\end{{{environment}}}',
        ]
    )


latex_column_labels = {
    'PV rated power': r'$P_{\mathrm{pv}}$ (kW)',
    'Wind rated power': r'$P_{\mathrm{wt}}$ (kW)',
    'Battery capacity': r'$B_{\mathrm{cap}}$ (kWh)',
    'TOPSIS coefficient': r'$c_i$',
}
latex_formats = {
    **{column: '.2f' for column in POSITION_COLUMNS},
    **{column: '.4f' for column in CRITERIA_COLUMNS},
    'TOPSIS coefficient': '.4f',
}
latex_table = dataframe_to_latex_table(
    compromise_table,
    caption='Equal-weight TOPSIS compromise solutions.',
    label='tab:topsis_compromise_solutions',
    column_labels=latex_column_labels,
    formats=latex_formats,
    column_format='ll' + 'r' * (len(compromise_table.columns) - 2),
)
print(latex_table)


## 5. Interactive objective and decision-space views

Both plots use exactly the same globally non-dominated alternatives. Colors identify the battery technology, and the highlighted diamond marks the equal-weight global TOPSIS solution. The Plotly figures can be rotated, zoomed, and inspected interactively.

In [ ]:
def plot_interactive_3d(
    frame,
    columns,
    axis_labels,
    selected=None,
    group_column='Battery code',
    group_labels=BATTERY_LABELS,
    group_colors=BATTERY_COLORS,
    selected_label='Global TOPSIS',
):
    """Create a rotatable three-dimensional Plotly scatter plot."""
    columns = list(columns)
    axis_labels = list(axis_labels)
    if len(columns) != 3 or len(axis_labels) != 3:
        raise ValueError('A three-dimensional plot requires three columns and labels.')

    fig = go.Figure()
    for group_value, group_label in group_labels.items():
        subset = frame[frame[group_column] == group_value]
        if subset.empty:
            continue
        fig.add_trace(
            go.Scatter3d(
                x=subset[columns[0]],
                y=subset[columns[1]],
                z=subset[columns[2]],
                mode='markers',
                marker={'size': 4, 'color': group_colors[group_value], 'opacity': 0.80},
                name=group_label,
                customdata=np.column_stack(
                    [subset['Battery'], subset[columns].to_numpy()]
                ),
                hovertemplate=(
                    'Battery: %{customdata[0]}<br>'
                    + f'{axis_labels[0]}: %{{customdata[1]:.4g}}<br>'
                    + f'{axis_labels[1]}: %{{customdata[2]:.4g}}<br>'
                    + f'{axis_labels[2]}: %{{customdata[3]:.4g}}<extra></extra>'
                ),
            )
        )

    if selected is not None:
        fig.add_trace(
            go.Scatter3d(
                x=[selected[columns[0]]],
                y=[selected[columns[1]]],
                z=[selected[columns[2]]],
                mode='markers',
                marker={
                    'size': 9, 'symbol': 'diamond', 'color': 'gold',
                    'line': {'color': 'black', 'width': 3},
                },
                name=selected_label,
                hovertemplate=(
                    f'{axis_labels[0]}: %{{x:.4g}}<br>'
                    + f'{axis_labels[1]}: %{{y:.4g}}<br>'
                    + f'{axis_labels[2]}: %{{z:.4g}}<extra></extra>'
                ),
            )
        )

    axis_style = {'backgroundcolor': 'white', 'gridcolor': 'lightgray'}
    fig.update_layout(
        scene={
            'xaxis': {**axis_style, 'title': axis_labels[0]},
            'yaxis': {**axis_style, 'title': axis_labels[1]},
            'zaxis': {**axis_style, 'title': axis_labels[2]},
            'camera': {'eye': {'x': 1.5, 'y': 1.5, 'z': 1.3}},
            'aspectmode': 'cube',
        },
        legend={
            'orientation': 'h',
            'x': 0.5, 'y': 0.01, 'xanchor': 'center', 'yanchor': 'bottom',
            'bgcolor': 'rgba(255,255,255,0.78)',
            'bordercolor': 'rgba(80,80,80,0.55)', 'borderwidth': 1,
            'font': {'size': 15}, 'itemsizing': 'constant',
        },
        margin={'l': 0, 'r': 0, 'b': 0, 't': 0},
        width=850,
        height=620,
        paper_bgcolor='white',
    )
    return fig


objective_space_figure = plot_interactive_3d(
    global_front,
    CRITERIA_COLUMNS,
    ['LCOE (USD/kWh)', 'RF', 'RSC'],
    selected=global_selected,
)
objective_space_figure.show()

decision_space_figure = plot_interactive_3d(
    global_front,
    POSITION_COLUMNS,
    ['PV rated power (kW)', 'Wind rated power (kW)', 'Battery capacity (kWh)'],
    selected=global_selected,
)
decision_space_figure.show()


## 6. Operational reconstruction of the global compromise

The global TOPSIS alternative is simulated again with its associated battery technology. The operational data frame is built directly from the simulation object, avoiding intermediate spreadsheet files and positional column indexing.

In [ ]:
SEASONAL_DATA_DIR = Path('../seasonal_data')
BATTERY_INDICES = {
    battery_code: index for index, battery_code in enumerate(BATTERY_LABELS)
}

OPERATION_COLUMNS = {
    'load': 'Load [kWh]',
    'pv_generation': 'Photovoltaic generation [kWh]',
    'wind_generation': 'Wind generation [kWh]',
    'pv_supply': 'Photovoltaic supply [kWh]',
    'wind_supply': 'Wind supply [kWh]',
    'battery_level': 'Battery energy level [kWh]',
    'battery_discharge': 'Battery discharge [kWh]',
    'battery_supply': 'Battery supply [kWh]',
    'purchased': 'Purchased energy [kWh]',
    'compensated': 'Compensated energy [kWh]',
    'surplus': 'Energy surplus [kWh]',
}


def load_seasonal_inputs(data_dir=SEASONAL_DATA_DIR):
    """Load and validate the four hourly input series."""
    inputs = {
        'load': np.genfromtxt(data_dir / 'load.txt'),
        'temperature': np.genfromtxt(data_dir / 'temperature.txt'),
        'solar_data': np.genfromtxt(data_dir / 'irradiance.txt'),
        'wind_data': np.genfromtxt(data_dir / 'wind.txt'),
    }
    lengths = {name: len(values) for name, values in inputs.items()}
    if len(set(lengths.values())) != 1:
        raise ValueError(f'Seasonal input lengths do not match: {lengths}')
    return inputs


def component_values(component, attribute, hours):
    """Read a simulated component series or return zeros when absent."""
    if component is None:
        return np.zeros(hours)
    values = np.asarray(getattr(component, attribute), dtype=float)
    if values.shape != (hours,):
        raise ValueError(f'{attribute} has shape {values.shape}; expected {(hours,)}.')
    return values


def build_operation_dataframe(microgrid):
    """Collect the simulated hourly flows without creating an Excel file."""
    hours = microgrid.hours
    frame = pd.DataFrame(
        {
            OPERATION_COLUMNS['load']: microgrid.load,
            OPERATION_COLUMNS['pv_generation']: component_values(
                microgrid.photovoltaic_panel, 'output_power', hours
            ),
            OPERATION_COLUMNS['wind_generation']: component_values(
                microgrid.wind_turbine, 'output_power', hours
            ),
            OPERATION_COLUMNS['pv_supply']: component_values(
                microgrid.photovoltaic_panel, 'meet_demand', hours
            ),
            OPERATION_COLUMNS['wind_supply']: component_values(
                microgrid.wind_turbine, 'meet_demand', hours
            ),
            OPERATION_COLUMNS['battery_level']: component_values(
                microgrid.battery, 'energy_level', hours
            ),
            OPERATION_COLUMNS['battery_discharge']: component_values(
                microgrid.battery, 'energy_discharged', hours
            ),
            OPERATION_COLUMNS['battery_supply']: component_values(
                microgrid.battery, 'meet_demand', hours
            ),
            OPERATION_COLUMNS['purchased']: component_values(
                microgrid.utility_grid, 'purchased_energy', hours
            ),
            OPERATION_COLUMNS['compensated']: component_values(
                microgrid.utility_grid, 'compensated_energy', hours
            ),
            OPERATION_COLUMNS['surplus']: microgrid.surplus_energy,
        }
    )
    frame.insert(0, 'Project hour', np.arange(hours))
    frame.insert(1, 'Project year', np.arange(hours) // microgrid.hours_per_interval + 1)
    frame.insert(2, 'Hour within year', np.arange(hours) % microgrid.hours_per_interval)
    return frame


def simulate_selected_solution(
    selected,
    position_columns=POSITION_COLUMNS,
    data_dir=SEASONAL_DATA_DIR,
):
    """Simulate one selected alternative and return the model and its log."""
    if len(position_columns) != 3:
        raise ValueError('The microgrid simulation requires three decision variables.')
    inputs = load_seasonal_inputs(data_dir)
    battery_code = selected['Battery code']
    microgrid = simulation(
        pv_rated_power=float(selected[position_columns[0]]),
        wt_rated_power=float(selected[position_columns[1]]),
        bat_capacity=float(selected[position_columns[2]]),
        select_bat=BATTERY_INDICES[battery_code],
        **inputs,
    )
    objectives = microgrid.run()
    operation = build_operation_dataframe(microgrid)
    return microgrid, objectives, operation


selected_microgrid, simulated_objectives, operation_data = (
    simulate_selected_solution(global_selected)
)


def validate_operational_reconstruction(
    selected, simulated_objectives, operation, absolute_tolerance=1e-8
):
    """Verify objective reproducibility and the hourly load balance."""
    delivered_to_load = sum(
        operation[column]
        for column in [
            OPERATION_COLUMNS['pv_supply'],
            OPERATION_COLUMNS['wind_supply'],
            OPERATION_COLUMNS['battery_supply'],
            OPERATION_COLUMNS['purchased'],
            OPERATION_COLUMNS['compensated'],
        ]
    )
    balance_residual = (
        operation[OPERATION_COLUMNS['load']] - delivered_to_load
    ).to_numpy()
    stored_objectives = selected[CRITERIA_COLUMNS].to_numpy(dtype=float)
    objective_residual = np.asarray(simulated_objectives) - stored_objectives

    diagnostics = pd.DataFrame(
        {
            'Diagnostic': [
                'Maximum hourly load-balance residual (kWh)',
                'Maximum objective reconstruction residual',
            ],
            'Value': [
                np.max(np.abs(balance_residual)),
                np.max(np.abs(objective_residual)),
            ],
        }
    )
    if not np.allclose(balance_residual, 0.0, atol=absolute_tolerance):
        raise RuntimeError('The reconstructed hourly energy balance is inconsistent.')
    if not np.allclose(
        simulated_objectives, stored_objectives,
        rtol=1e-8, atol=absolute_tolerance,
    ):
        raise RuntimeError('The selected objectives were not reproduced by simulation.')
    return diagnostics


validation_diagnostics = validate_operational_reconstruction(
    global_selected, simulated_objectives, operation_data
)
print('Re-simulated objectives:', simulated_objectives)
display(validation_diagnostics)
display(operation_data.head())


## 7. Interactive operational views

The monthly stacked bar chart is preferred to a pie chart because it retains seasonal information while showing how the demand is supplied.

In [ ]:
SUPPLY_SERIES = [
    ('Photovoltaic', OPERATION_COLUMNS['pv_supply'], '#F2C94C'),
    ('Wind', OPERATION_COLUMNS['wind_supply'], '#27AE60'),
    ('Battery', OPERATION_COLUMNS['battery_supply'], '#2F80ED'),
    ('Grid purchase', OPERATION_COLUMNS['purchased'], '#EB5757'),
    ('Grid compensation', OPERATION_COLUMNS['compensated'], '#F2994A'),
]


def select_project_year(operation, project_year, hours_per_year):
    """Select one project year and attach a reference calendar."""
    yearly = operation[operation['Project year'] == project_year].copy()
    if yearly.empty:
        raise ValueError(f'Project year {project_year} is unavailable.')
    if len(yearly) != hours_per_year:
        raise ValueError('The selected project year is incomplete.')
    if hours_per_year == 8784:
        reference_start = '2024-01-01'
    elif hours_per_year == 8760:
        reference_start = '2023-01-01'
    else:
        raise ValueError('Monthly views require a 8760- or 8784-hour year.')
    yearly['Reference time'] = pd.date_range(
        reference_start, periods=hours_per_year, freq='h'
    )
    return yearly


def plot_operation(
    operation,
    hours_per_year,
    project_year=1,
    start_hour=0,
    hours=168,
):
    """Plot an interactive hourly operating window."""
    yearly = select_project_year(operation, project_year, hours_per_year)
    window = yearly.iloc[start_hour:start_hour + hours]
    if window.empty:
        raise ValueError('The selected operating window is empty.')

    fig = go.Figure()
    for label, column, color in SUPPLY_SERIES:
        fig.add_trace(
            go.Bar(
                x=window['Reference time'], y=window[column],
                name=label, marker_color=color,
                hovertemplate='%{x|%b %d, %H:%M}<br>%{y:.3f} kWh<extra></extra>',
            )
        )
    fig.add_trace(
        go.Scatter(
            x=window['Reference time'],
            y=window[OPERATION_COLUMNS['load']],
            mode='lines', name='Load',
            line={'color': '#6C3483', 'width': 2.5},
        )
    )
    fig.add_trace(
        go.Scatter(
            x=window['Reference time'],
            y=window[OPERATION_COLUMNS['battery_level']],
            mode='lines', name='Battery energy level', yaxis='y2',
            line={'color': '#154360', 'width': 2, 'dash': 'dash'},
        )
    )
    fig.update_layout(
        barmode='stack',
        xaxis_title='Reference time',
        yaxis={'title': 'Energy per time step (kWh)'},
        yaxis2={
            'title': 'Battery energy level (kWh)', 'overlaying': 'y', 'side': 'right'
        },
        hovermode='x unified',
        legend={'orientation': 'h', 'y': -0.22, 'font': {'size': 18}},
        margin={'l': 60, 'r': 70, 'b': 100, 't': 30},
        width=1000, height=560,
    )
    return fig


def plot_monthly_supply(operation, hours_per_year, project_year=1):
    """Plot the monthly demand-supply balance as an interactive bar chart."""
    yearly = select_project_year(operation, project_year, hours_per_year)
    value_columns = [column for _, column, _ in SUPPLY_SERIES]
    value_columns.append(OPERATION_COLUMNS['load'])
    monthly = (
        yearly.set_index('Reference time')[value_columns]
        .resample('MS').sum() / 1000
    )
    month_labels = monthly.index.strftime('%b')

    fig = go.Figure()
    for label, column, color in SUPPLY_SERIES:
        fig.add_trace(
            go.Bar(x=month_labels, y=monthly[column], name=label, marker_color=color)
        )
    fig.add_trace(
        go.Scatter(
            x=month_labels, y=monthly[OPERATION_COLUMNS['load']],
            mode='lines+markers', name='Load',
            line={'color': '#6C3483', 'width': 3},
        )
    )
    fig.update_layout(
        barmode='stack', xaxis_title='Month', yaxis_title='Energy (MWh)',
        hovermode='x unified',
        legend={'orientation': 'h', 'y': -0.22, 'font': {'size': 18}},
        margin={'l': 60, 'r': 30, 'b': 100, 't': 30}, width=900, height=520,
    )
    return fig


operation_figure = plot_operation(
    operation_data, selected_microgrid.hours_per_interval,
    project_year=1, start_hour=3932, hours=360,
)
operation_figure.show()

monthly_supply_figure = plot_monthly_supply(
    operation_data, selected_microgrid.hours_per_interval, project_year=1
)
monthly_supply_figure.show()


## 8. TOPSIS weight sensitivity

The sensitivity analysis varies the relative importance of LCOE, RF, and RSC over the complete three-criterion weight simplex while retaining the same global non-dominated front. It therefore evaluates preference sensitivity, rather than uncertainty in the simulation inputs or local perturbations of the selected capacities. A suitable associated research question is: *How sensitive are the selected battery technology and compromise design to changes in the relative importance assigned to LCOE, RF, and RSC?*

The weight-grid shares reported below describe geometric coverage of the sampled preference simplex; they must not be interpreted as probabilities of selection. Exogenous economic, demand, and resource uncertainties are reserved for future work.

In [ ]:
TOPSIS_SENSITIVITY_RESOLUTION = 42  # Weight increment = 1 / TOPSIS_SENSITIVITY_RESOLUTION


def three_criterion_weight_simplex(resolution=42):
    """Generate an exact uniform grid over a three-criterion weight simplex."""
    if not isinstance(resolution, int) or resolution < 1:
        raise ValueError('resolution must be a positive integer.')
    integer_weights = [
        (first, second, resolution - first - second)
        for first in range(resolution + 1)
        for second in range(resolution - first + 1)
    ]
    return np.asarray(integer_weights, dtype=float) / resolution


def topsis_weight_sensitivity(
    front,
    criteria_columns,
    directions,
    position_columns=POSITION_COLUMNS,
    resolution=TOPSIS_SENSITIVITY_RESOLUTION,
):
    """Select the best alternative for every point of a ternary weight grid."""
    criteria_columns = list(criteria_columns)
    if len(criteria_columns) != 3:
        raise ValueError('Ternary sensitivity requires exactly three criteria.')

    candidates = front.reset_index(drop=True).copy()
    candidates.insert(0, 'Alternative index', np.arange(len(candidates)))
    weight_columns = [f'Weight {criterion}' for criterion in criteria_columns]
    records = []

    for weights in three_criterion_weight_simplex(resolution):
        ranking = topsis_rank(
            candidates, criteria_columns, weights, directions
        )
        selected = ranking.iloc[0]
        tied = np.isclose(
            ranking['TOPSIS coefficient'],
            selected['TOPSIS coefficient'],
            rtol=1e-10, atol=1e-12,
        )
        record = {
            **dict(zip(weight_columns, weights)),
            'Alternative index': int(selected['Alternative index']),
            'Battery code': selected['Battery code'],
            'Battery': selected['Battery'],
            'TOPSIS coefficient': selected['TOPSIS coefficient'],
            'Number of tied alternatives': int(tied.sum()),
        }
        for column in [*position_columns, *criteria_columns]:
            record[column] = selected[column]
        records.append(record)

    return pd.DataFrame(records)


def summarize_weight_sensitivity(sensitivity):
    """Summarize technology coverage over the sampled weight simplex."""
    summary = (
        sensitivity.groupby(['Battery code', 'Battery'], sort=False)
        .agg(
            **{
                'Grid points': ('Alternative index', 'size'),
                'Distinct alternatives': ('Alternative index', 'nunique'),
            }
        )
        .reset_index()
    )
    summary['Grid share (%)'] = 100 * summary['Grid points'] / len(sensitivity)
    return summary.sort_values('Grid points', ascending=False).reset_index(drop=True)


def plot_weight_sensitivity_ternary(
    sensitivity,
    criteria_columns=CRITERIA_COLUMNS,
    group_labels=BATTERY_LABELS,
    group_colors=BATTERY_COLORS,
):
    """Plot the selected battery technology across the weight simplex."""
    weight_columns = [f'Weight {criterion}' for criterion in criteria_columns]
    fig = go.Figure()

    for battery_code, battery_label in group_labels.items():
        subset = sensitivity[sensitivity['Battery code'] == battery_code]
        if subset.empty:
            continue
        custom_columns = [
            'Alternative index', *POSITION_COLUMNS, *CRITERIA_COLUMNS,
            'Number of tied alternatives',
        ]
        fig.add_trace(
            go.Scatterternary(
                a=subset[weight_columns[0]],
                b=subset[weight_columns[1]],
                c=subset[weight_columns[2]],
                mode='markers', name=battery_label, cliponaxis=False,
                marker=go.scatterternary.Marker(
                    size=9, color=group_colors[battery_code], opacity=1.0
                ),
                customdata=subset[custom_columns].to_numpy(),
                hovertemplate=(
                    f'{battery_label}<br>'
                    + f'w({criteria_columns[0]})=%{{a:.3f}}<br>'
                    + f'w({criteria_columns[1]})=%{{b:.3f}}<br>'
                    + f'w({criteria_columns[2]})=%{{c:.3f}}<br>'
                    + 'Alternative=%{customdata[0]:.0f}<br>'
                    + 'PV=%{customdata[1]:.3f} kW<br>'
                    + 'Wind=%{customdata[2]:.3f} kW<br>'
                    + 'Battery=%{customdata[3]:.3f} kWh<br>'
                    + 'LCOE=%{customdata[4]:.5f}<br>'
                    + 'RF=%{customdata[5]:.5f}<br>'
                    + 'RSC=%{customdata[6]:.5f}<br>'
                    + 'Tied alternatives=%{customdata[7]:.0f}'
                    + '<extra></extra>'
                ),
            )
        )

    fig.add_trace(
        go.Scatterternary(
            a=[1 / 3], b=[1 / 3], c=[1 / 3], mode='markers',
            marker=go.scatterternary.Marker(
                size=12, symbol='diamond', color='gold',
                line=go.scatterternary.marker.Line(color='black', width=2),
            ),
            name='Equal weights',
            hovertemplate='Equal weights<extra></extra>',
        )
    )
    fig.update_layout(
        ternary=go.layout.Ternary(
            sum=1, bgcolor='white',
            domain=go.layout.ternary.Domain(x=[0.02, 0.98], y=[0.04, 0.99]),
            aaxis=go.layout.ternary.Aaxis(
                title=f'w({criteria_columns[0]}) = 1', min=0,
                tick0=0, dtick=0.2, tickformat='.1f',
                ticks='outside', ticklen=10, layer='below traces',
                linecolor='#666666', gridcolor='#D9D9D9',
            ),
            baxis=go.layout.ternary.Baxis(
                title=f'w({criteria_columns[1]}) = 1', min=0,
                tick0=0, dtick=0.2, tickformat='.1f',
                ticks='outside', ticklen=10, layer='below traces',
                linecolor='#666666', gridcolor='#D9D9D9',
            ),
            caxis=go.layout.ternary.Caxis(
                title=f'w({criteria_columns[2]}) = 1', min=0,
                tick0=0, dtick=0.2, tickformat='.1f',
                ticks='outside', ticklen=10, layer='below traces',
                linecolor='#666666', gridcolor='#D9D9D9',
            ),
        ),
        legend=go.layout.Legend(
            orientation='h', x=0.5, y=-0.02,
            xanchor='center', yanchor='top',
            bgcolor='rgba(255,255,255,0.80)',
            font=go.layout.legend.Font(size=15),
        ),
        margin=go.layout.Margin(l=30, r=30, b=55, t=30),
        width=900, height=700, paper_bgcolor='white',
    )
    return fig


weight_sensitivity = topsis_weight_sensitivity(
    global_front,
    CRITERIA_COLUMNS,
    CRITERIA_DIRECTIONS,
    resolution=TOPSIS_SENSITIVITY_RESOLUTION,
)
weight_sensitivity_summary = summarize_weight_sensitivity(weight_sensitivity)

equal_weight_match = np.all(
    np.isclose(
        global_front[[*POSITION_COLUMNS, *CRITERIA_COLUMNS]].to_numpy(),
        global_selected[[*POSITION_COLUMNS, *CRITERIA_COLUMNS]].to_numpy(dtype=float),
    ),
    axis=1,
)
equal_weight_index = int(np.flatnonzero(equal_weight_match)[0])
same_alternative_share = 100 * np.mean(
    weight_sensitivity['Alternative index'] == equal_weight_index
)
same_battery_share = 100 * np.mean(
    weight_sensitivity['Battery code'] == global_selected['Battery code']
)

print(f'Weight-grid points: {len(weight_sensitivity)}')
print(f'Distinct selected alternatives: {weight_sensitivity["Alternative index"].nunique()}')
print(f'Equal-weight alternative grid share: {same_alternative_share:.2f}%')
print(f'Equal-weight battery grid share: {same_battery_share:.2f}%')
display(weight_sensitivity_summary)

weight_sensitivity_latex = dataframe_to_latex_table(
    weight_sensitivity_summary[
        ['Battery', 'Grid points', 'Grid share (%)', 'Distinct alternatives']
    ],
    caption='TOPSIS selections over the criterion-weight simplex.',
    label='tab:topsis_weight_sensitivity',
    column_labels={'Grid share (%)': r'Grid share (\%)'},
    formats={'Grid share (%)': '.2f'},
    column_format='lrrr',
    environment='table',
)
print(weight_sensitivity_latex)

weight_sensitivity_figure = plot_weight_sensitivity_ternary(weight_sensitivity)
weight_sensitivity_figure.show()


# Benchmark 2D:

In [ ]:
objective_dim = 2 # Number of objectives
position_dim = 20 # Number of variables
n_pareto_points = 5000 # Number of Pareto points to be generated
wfg_k = 6 # The parameter k for WFG problems

experiment_name = "zdt2"

# Choosing the files
insert_maco = False
insert_mopso_cd = False
insert_nsga2 = False
insert_nspso = False
insert_spea2 = False


file_tuple = []

# A-MESH files
file_tuple.append((f"AMESH_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'A-MESH'))
 
if insert_maco:
    file_tuple.append((f'MACO_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'MACO'))
if insert_mopso_cd:
    file_tuple.append((f'MOPSO_CD_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'MOPSO-CD'))
if insert_nsga2:
  	file_tuple.append((f'NSGA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II'))
if insert_nspso:
    file_tuple.append((f'NSPSO_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSPSO'))
if insert_spea2:
    file_tuple.append((f'SPEA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'SPEA-II'))

num_datasets = len(file_tuple) # Number of dataset

##### Color setup #####
cmap = plt.get_cmap("tab10")
solid_colors = [mcolors.to_hex(cmap(i+1)) for i in range(num_datasets)]
colorscales = pc.named_colorscales() # List of color scales
#######################

##### Marker setup #####
marker_symbols = ['circle', 'diamond', 'square', 'cross', 'x', 'circle-open', 'diamond-open', 'square-open']
num_symbols = len(marker_symbols)
#######################

pareto_front = []
for i in range(num_datasets):
  with open("../results/" + file_tuple[i][0], "rb") as f:
    r = pickle.load(f)
    pareto_front.append(
      go.Scatter(x=r['combined'][1][:,0],
                   y=r['combined'][1][:,1],
                   mode='markers', 
                   marker=dict(size=8, symbol=marker_symbols[i % num_symbols], color=solid_colors[i]), #r['combined'][1][:,1], colorscale=colorscales[(i+1) % num_scales]),
                   name=file_tuple[i][1],
                   showlegend=True)
    )

## Plotting

In [ ]:
# Creating a subplot
fig = go.Figure()

for i in range(num_datasets):
    fig.add_trace(pareto_front[i])

axis_range = None # [-0.05, 1.05]
# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"  # Transparente
        )
    ],
    xaxis=dict(title=dict(text='f1', font=dict(size=14)), tickfont=dict(size=14)),
    yaxis=dict(title=dict(text='f2', font=dict(size=14)), tickfont=dict(size=14)),
    # title=dict(
    #     text='A-MESH - ' + experiment_name.upper(),
    #     x=0.5,  # Centering the title
    #     xanchor='center',
    #     yanchor='top'
    # ),
    legend=dict(
        x=0.99,
        y=0.99,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",  # fundo leve para não colidir com os pontos
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    ),
    showlegend=True,
    margin=dict(l=5, r=5, b=5, t=5), # Rearrange the margin
    width=600,   # Figure width (pixels)
    height=480,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Plotting the Pareto front
real_pareto_front = get_pareto(experiment_name, n_pareto_points, position_dim, 2, wfg_k)
fig.add_trace(
	go.Scatter(x=real_pareto_front[:,0],
		y=real_pareto_front[:,1],
		mode='markers', marker=dict(size=2, color='black'),
		name='Pareto Front',
		showlegend=True
		)
	)

# Showing the figure
fig.show()

# Benchmark 3D:

In [ ]:
objective_dim = 3 # Number of objectives
position_dim = 20 # Number of variables
n_pareto_density = 100 # Number of Pareto points to be generated
wfg_k = 6 # The parameter k for WFG problems

experiment_name = "dtlz1"

# Choosing the files
insert_maco = False
insert_mopso_cd = False
insert_nsga2 = False
insert_nspso = False
insert_spea2 = False


file_tuple = []

# A-MESH files
file_tuple.append((f"AMESH_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'A-MESH'))

if insert_maco:
    file_tuple.append((f'MACO_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'MACO'))
if insert_mopso_cd:
    file_tuple.append((f'MOPSO_CD_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'MOPSO-CD'))
if insert_nsga2:
  	file_tuple.append((f'NSGA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II'))
if insert_nspso:
    file_tuple.append((f'NSPSO_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSPSO'))
if insert_spea2:
    file_tuple.append((f'SPEA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'SPEA-II'))

num_datasets = len(file_tuple) # Number of dataset

##### Color setup #####
cmap = plt.get_cmap("tab10")
solid_colors = [mcolors.to_hex(cmap(i+1)) for i in range(num_datasets)]
colorscales = pc.named_colorscales() # List of color scales
#######################

##### Marker setup #####
marker_symbols = ['circle', 'diamond', 'square', 'cross', 'x', 'circle-open', 'diamond-open', 'square-open']
num_symbols = len(marker_symbols)
#######################

pareto_front = []
for i in range(num_datasets):
  with open("../results/" + file_tuple[i][0], "rb") as f:
    r = pickle.load(f)
    pareto_front.append(
      go.Scatter3d(x=r['combined'][1][:,0],
                   y=r['combined'][1][:,1],
                   z=r['combined'][1][:,2],
                   mode='markers', marker=dict(size=5, symbol=marker_symbols[i % num_symbols], color=solid_colors[i]), #r['combined'][1][:,1], colorscale=colorscales[(i+1) % num_scales]),
                   name=file_tuple[i][1],
                   showlegend=True)
    )

## Plotting

In [ ]:
# Creating a subplot
fig = go.Figure()

for i in range(num_datasets):
    fig.add_trace(pareto_front[i])

axis_range = None # [-0.1, 1.2]

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"  # Transparente
        )
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=1,y=1,z=1)
        ),
        xaxis=dict(
            title=dict(text='f1', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # X-axis background
            gridcolor='lightgray',    # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='f2', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # Y-axis background
            gridcolor='lightgray',    # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='f3', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # Z-axis background
            gridcolor='lightgray',    # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        aspectmode='cube'
    ),
    legend=dict(
        x=0.99,
        y=0.99,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",  # fundo leve para não colidir com os pontos
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    ),
    showlegend=True,
    # title=dict(
    #     text='A-MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=5, r=5, b=5, t=5), # Rearrange the margin
    width=600,   # Figure width (pixels)
    height=420,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Plotting the Pareto front
real_pareto_front = get_pareto(experiment_name, n_pareto_density, position_dim, 3, wfg_k)
fig.add_trace(
	go.Scatter3d(x=real_pareto_front[:,0],
		y=real_pareto_front[:,1],
		z=real_pareto_front[:,2],
		mode='markers', marker=dict(size=2, color='black'),
		name='Pareto Front',
		showlegend=True
		)
	)

# Showing the figure
fig.show()